# Study 3 on Google Colab

Runs the locked `docs/PREREGISTRATION_STUDY3.md`. It adds 419 benign rows to PIDS-Bench's training set in 4 ways, plus one secondary way, and measures how often each detector wrongly flags real, security-adjacent benign prompts.

**Before you start**
1. *Runtime → Change runtime type → A100 GPU.*
2. You must have accepted the LMSYS-Chat-1M licence on Hugging Face, and your read token must be in Colab's 🔑 *Secrets* panel as `HF_TOKEN` with notebook access on. **Never paste the token into a cell.**
3. The project owner's approval line must already be in `AGENTS.md`. Otherwise the runner refuses to start.

**LMSYS licence.** LMSYS text is written only to `MyDrive/study3/private_lmsys_text_do_not_share`. Never share, upload or commit that folder. Only `MyDrive/study3/public` is meant to come back to the repo; it holds fingerprints, counts and scores, no text.

**Order.** Run the pilot cell first. It builds the data pools (streaming LMSYS takes a while) and trains one model, then prints how long the fit took. Multiply that by 25 to estimate the whole run before continuing. Progress is saved to Drive after every model. If Colab disconnects, run all cells again and it continues where it stopped; only the interrupted model is redone.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/study3'

In [ ]:
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # from Secrets; never printed
if not os.path.exists('/content/prompt_injection'):
    os.system('git clone -q -b claude/vigilant-fermi-r2ovyf https://github.com/dannyg26/prompt_injection /content/prompt_injection')
os.chdir('/content/prompt_injection')
!git pull -q origin claude/vigilant-fermi-r2ovyf
!git log --oneline -1

In [ ]:
# Our pinned stack, then PIDS-Bench's pins for its training code (their requirements.txt).
!pip install -q -r requirements-lock.txt
!pip install -q transformers==4.57.1 datasets==4.4.1 python-dotenv==1.0.1 pandas sentencepiece protobuf
!pip install -q --no-deps -e .
!python -c "import torch, transformers, datasets; print(torch.__version__, transformers.__version__, datasets.__version__)"

## 1. Pilot: builds the pools and trains one model, then prints the time

In [ ]:
!PYTHONPATH=src python scripts/run_study3.py --out "$OUT" --pilot 2>&1 | grep -v 'SemanticDedup\] processed' | tail -n 40

## 2. Full run (resumable; reuses the pilot's model)

In [ ]:
!PYTHONPATH=src python scripts/run_study3.py --out "$OUT" 2>&1 | grep '\[study3' ; tail -c 3000 "$OUT/public/study3_results.json"

## After it finishes

Download these two files and send them back:
- `MyDrive/study3/public/study3_results.json`
- `MyDrive/study3/public/pool_manifest.json`

Keep `public/scores/` and `public/logs/` on Drive for the audit trail. Do **not** send anything from `private_lmsys_text_do_not_share`.

In [ ]:
from google.colab import files
files.download(f'{OUT}/public/study3_results.json')
files.download(f'{OUT}/public/pool_manifest.json')